# Run Eval — score saved conversations with the oracle

Per-patient eval CSVs co-locate with the run that produced them, labelled by the
scoring **metric** and the training **oracle**:
`data/<method>/eval_scores/metric=<Q>/oracle=<O>/<Model>/<patient_id>.csv`
(Exp2 baselines → `pto_Exp2`, GRPO_Exp3 → `grpo_Exp3`, PTO_Exp3 → `pto_Exp3`;
`oracle=none` for the untrained Base). Each model's root + oracle are resolved
from the registry via `get_model_eval_layout()`. Resume-safe: existing CSVs are
skipped, so re-running picks up where you left off.

Run [Conv_EDA.ipynb](Conv_EDA.ipynb) after this notebook completes.

## 1. Setup

In [ ]:
from lib import (
    WORKSPACE_ROOT, EDAConfig, DATA_DIR, EVAL_QUESTIONNAIRE_DIRS, eval_csv_dir,
    DEFAULT_CONCURRENCY, EVAL_MODEL, EVAL_TEMPERATURE,
    get_model_names, get_model_eval_layout, resolve_paths,
    load_data, combine_data, add_model_metadata_columns,
)
from lib import eval as oracle_eval

import os
import pandas as pd
from IPython.display import display

assert oracle_eval.EVAL_CODE_AVAILABLE, (
    "questionnaires module not on sys.path — "
    "verify Exp3_PTO_GRPO/code/ is reachable."
)

## 2. Knobs

`QUESTIONNAIRE_FILTER`: subset of `{"Q1", "Q2", "WAI-SR", "CSQ-8", "MI-SAT", "MITI"}` to score (None = all six).
`EXPERIMENT_GROUP_FILTER`: subset of ExperimentGroups (Base always included). None = every model on disk.


In [ ]:
# ┌─ Configuration ───────────────────────────────────────────────────
QUESTIONNAIRE_FILTER    = None
EXPERIMENT_GROUP_FILTER = None
CONCURRENCY             = DEFAULT_CONCURRENCY
# └───────────────────────────────────────────────────────────────────

config = EDAConfig(async_concurrency=CONCURRENCY)

# Eval scores co-locate per method, labelled by metric + training oracle:
#   data/<method>/eval_scores/metric=<M>/oracle=<O>/<model>/   (resolved from the registry)
MODEL_LAYOUT = get_model_eval_layout()

print(f"Eval model      : {config.eval_model}")
print(f"Eval temperature: {config.eval_temp}")
print(f"Concurrency     : {config.async_concurrency}")
print("Eval roots in use:")
for _root in sorted({v["root"] for v in MODEL_LAYOUT.values()}):
    print(f"  → {_root}")

## 3. Load Conversations

In [ ]:
combined_data = combine_data(load_data(resolve_paths()), get_model_names())
combined_data["id"] = combined_data["id"].astype(int)
combined_data = add_model_metadata_columns(combined_data)

if EXPERIMENT_GROUP_FILTER is not None:
    keep = set(EXPERIMENT_GROUP_FILTER) | {"Base"}
    combined_data = combined_data[combined_data["ExperimentGroup"].isin(keep)].copy()

print(
    f"{len(combined_data):,} conversations · "
    f"{combined_data['Model'].nunique()} models · "
    f"{combined_data['ExperimentGroup'].nunique()} experiment groups"
)


## 4. Coverage Summary — what's done, what's missing

For each (model × questionnaire) combo: how many eval CSVs already exist vs how many conversations
are on disk. `_todo` columns sum to the total scoring calls the next cell will make.
Coverage matrix is also saved to `data/eval_scores/coverage.csv` for later reference.


In [ ]:
active_q = (
    list(EVAL_QUESTIONNAIRE_DIRS) if QUESTIONNAIRE_FILTER is None
    else [q for q in EVAL_QUESTIONNAIRE_DIRS if q in QUESTIONNAIRE_FILTER]
)

rows = []
for model in sorted(combined_data["Model"].astype(str).unique()):
    n_conv = int((combined_data["Model"] == model).sum())
    row = {"Model": model, "convs": n_conv}
    entry = MODEL_LAYOUT.get(model)
    for qname in active_q:
        qf = eval_csv_dir(entry["root"], entry["oracle"], EVAL_QUESTIONNAIRE_DIRS[qname], model) if entry else None
        done = len([f for f in os.listdir(qf) if f.endswith(".csv")]) if qf and os.path.isdir(qf) else 0
        row[f"{qname}_done"] = done
        row[f"{qname}_todo"] = max(0, n_conv - done)
    rows.append(row)

coverage_df = pd.DataFrame(rows).set_index("Model")
todo_cols = [c for c in coverage_df.columns if c.endswith("_todo")]
total_todo = int(coverage_df[todo_cols].to_numpy().sum())
print(f"Total scoring calls to make: {total_todo:,}")

# Persist coverage matrix to data/ for later reference.
os.makedirs(DATA_DIR, exist_ok=True)
coverage_path = os.path.join(DATA_DIR, "eval_coverage.csv")
coverage_df.to_csv(coverage_path)
print(f"Saved coverage matrix → {coverage_path}")

display(coverage_df.style.set_caption(
    f"Eval coverage — {len(coverage_df)} models × {len(active_q)} questionnaires"
))

## 5. Run Async Eval

In [ ]:
from openai import AsyncOpenAI
with open(os.path.join(WORKSPACE_ROOT, "openai_key.txt")) as fh:
    _client = AsyncOpenAI(api_key=fh.read().strip())

_configs = oracle_eval.build_default_eval_configs(config)
if QUESTIONNAIRE_FILTER is not None:
    _configs = [c for c in _configs if c["name"] in QUESTIONNAIRE_FILTER]

stats = await oracle_eval.run_all_evaluations_async(
    _client, combined_data, _configs, MODEL_LAYOUT, concurrency=config.async_concurrency,
)

display(pd.DataFrame(stats).T.style.set_caption("Run summary — per questionnaire"))